
# Transform Results Data

1. Read bronze results table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (constructorId -> constructor_id, driverId -> driver_id, reaceNmae -> race_name, positionText -> finish_position_text)
4. Rename Columns to make them more meaningful (date -> race_date, grid -> grid_positions, laps -> completed_laps, number -> car_number, position -> finish_position)
5. Filter out rows where season, round, constructor_id or driver_id is null (business key validation)
6. Remove duplicate records
7. Transform values of columns race_name to Title Case
8. Write the transformed data to silver results table

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
%run ../00-Common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"


### Step 1 - Read bronze constructors table

In [0]:
# circuits_df = spark.table(bronze_table)
results_df = spark.read.table(bronze_table)

### Step 2 - Keep only the columns required for analytics (Drop url column)



In [0]:
results_selected_df = results_df.drop(
    col("url")   
)


### Step 3 - 
- Standardise column names using snake_case (constructorId -> constructor_id, driverId -> driver_id, reaceNmae -> race_name, positionText -> finish_position_text)
- Rename Columns to make them more meaningful (date -> race_date, grid -> grid_positions, laps -> completed_laps, number -> car_number, position -> finish_position)

In [0]:
results_renamed_df = results_selected_df\
    .withColumnsRenamed(
        {
            "constructorId":"constructor_id",
            "driverId":"driver_id",
            "raceName":"race_name",
            "date":"race_date",
            "grid":"grid_position",
            "laps":"completed_laps",
            "number":"car_number",
            "position":"final_position",
            "positionText":"final_position_text"
        }
    )

### Step 5 - Filter out rows where season, round, constructor_id or driver_id is null (business key validation)

In [0]:
results_valid_df = results_renamed_df.filter(
    col("season").isNotNull() &
    col("round").isNotNull() &
    col("constructor_id").isNotNull() &
    col("driver_id").isNotNull() 
)


### Step 6 - Remove Duplicate Records


In [0]:
results_distinct_df = results_valid_df.dropDuplicates(["season","round","constructor_id","driver_id"])


### Step  7 - Transform Value of column nationality to Title Case


In [0]:
results_final_df = (
    results_distinct_df
    .withColumn('race_name', initcap('race_name'))
)



### Step  8 - Write the transformed data to silver results table

In [0]:
(
    results_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))